# EnergyAware AI - Step 1: Data collection

Runs 800 queries (400 GSM8K, 200 MMLU, 200 ARC-Challenge) through three Qwen2.5 tiers (0.5B, 3B, 7B in 4-bit) and logs correctness, latency and measured GPU energy for every (query, tier) pair.

Needs a **GPU runtime**. Results are saved to `MyDrive/energyaware` and the run **resumes** after a disconnect. If you see a `sympy` import error after the install cell, choose Runtime -> Restart session and run from the top again.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes datasets nvidia-ml-py pandas

In [ ]:
import os, re, gc, json, time, random, threading
import torch, pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ---------------- CONFIG ----------------
SEED = 42
N_GSM = 400                 # GSM8K queries: the first 200 are already collected, 200 new ones will run
N_MC = 200                  # MMLU and ARC queries (unchanged)
MAX_NEW_GSM = 320
MAX_NEW_MC = 8

# Ordered cheapest -> most expensive. This order is used for "oracle routing" later.
TIERS = [
    {"name": "small",  "id": "Qwen/Qwen2.5-0.5B-Instruct", "quant4": False},
    {"name": "medium", "id": "Qwen/Qwen2.5-3B-Instruct",   "quant4": False},
    {"name": "large",  "id": "Qwen/Qwen2.5-7B-Instruct",   "quant4": True},
]

# Where results are saved (survives restarts if on Drive / Kaggle working dir)
if os.path.exists("/kaggle/working"):
    OUT_DIR = "/kaggle/working/energyaware"
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = "/content/drive/MyDrive/energyaware"
    except Exception:
        OUT_DIR = "./energyaware"
os.makedirs(OUT_DIR, exist_ok=True)
QUERIES_PATH = f"{OUT_DIR}/queries.jsonl"
RESULTS_PATH = f"{OUT_DIR}/results.jsonl"
print("Saving to:", OUT_DIR)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU runtime!")

## 1. GPU power meter
Samples GPU power (watts) in a background thread via NVML and integrates it over time to get joules.
We also measure **idle power** so we can report *net* energy (energy above idle).

In [ ]:
import pynvml
pynvml.nvmlInit()
_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)

def gpu_watts():
    return pynvml.nvmlDeviceGetPowerUsage(_HANDLE) / 1000.0   # mW -> W

class PowerMeter:
    """Context manager: samples GPU power and integrates to joules (trapezoid rule)."""
    def __init__(self, interval=0.02):
        self.interval = interval
        self.samples = []
        self._stop = threading.Event()
    def _run(self):
        while not self._stop.is_set():
            self.samples.append((time.perf_counter(), gpu_watts()))
            time.sleep(self.interval)
    def __enter__(self):
        self.samples = []
        self._stop.clear()
        self._t = threading.Thread(target=self._run, daemon=True)
        self._t.start()
        return self
    def __exit__(self, *a):
        self._stop.set()
        self._t.join()
        s = self.samples
        self.energy_j = 0.0
        for (t0, w0), (t1, w1) in zip(s, s[1:]):
            self.energy_j += 0.5 * (w0 + w1) * (t1 - t0)
        self.mean_w = sum(w for _, w in s) / len(s) if s else float("nan")

def measure_idle_watts(seconds=3.0):
    vals = []
    t_end = time.time() + seconds
    while time.time() < t_end:
        vals.append(gpu_watts()); time.sleep(0.05)
    return sum(vals) / len(vals)

print("Idle GPU power (W):", round(measure_idle_watts(), 1))

## 2. Build the query set
Three datasets with **checkable answers** (no LLM-as-judge needed): GSM8K (math), MMLU (knowledge), ARC-Challenge (science reasoning).
Data is shuffled with a fixed seed and the first N are taken, so raising N later keeps the earlier queries (and resumes cleanly).

In [ ]:
LETTERS = "ABCD"

def mc_prompt(q, choices):
    opts = "\n".join(f"{LETTERS[i]}. {c}" for i, c in enumerate(choices))
    return f"{q}\n\n{opts}\n\nAnswer with only the letter of the correct option."

def gsm_prompt(q):
    return f"{q}\n\nSolve step by step. End your reply with 'Answer: <number>'."

def build_queries(n_gsm, n_mc):
    rows = []
    rng = random.Random(SEED)

    g = load_dataset("openai/gsm8k", "main", split="test").shuffle(seed=SEED).select(range(n_gsm))
    for i, r in enumerate(g):
        gold = r["answer"].split("####")[-1].strip().replace(",", "")
        rows.append({"qid": f"gsm8k_{i}", "dataset": "gsm8k", "type": "math",
                     "prompt": gsm_prompt(r["question"]), "gold": gold})

    m = load_dataset("cais/mmlu", "all", split="test").shuffle(seed=SEED).select(range(n_mc))
    for i, r in enumerate(m):
        rows.append({"qid": f"mmlu_{i}", "dataset": "mmlu", "type": "mc",
                     "prompt": mc_prompt(r["question"], r["choices"]), "gold": LETTERS[r["answer"]]})

    a = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test").shuffle(seed=SEED)
    k = 0
    for r in a:
        labels, texts = r["choices"]["label"], r["choices"]["text"]
        if len(texts) != 4 or r["answerKey"] not in labels:
            continue
        rows.append({"qid": f"arc_{k}", "dataset": "arc", "type": "mc",
                     "prompt": mc_prompt(r["question"], texts),
                     "gold": LETTERS[labels.index(r["answerKey"])]})
        k += 1
        if k >= n_mc:
            break
    return rows

queries = build_queries(N_GSM, N_MC)
with open(QUERIES_PATH, "w") as f:
    for q in queries:
        f.write(json.dumps(q) + "\n")
print(len(queries), "queries")
print(pd.Series([q["dataset"] for q in queries]).value_counts().to_dict())

## 3. Model loading, generation, and answer checking

In [ ]:
def load_model(spec):
    tok = AutoTokenizer.from_pretrained(spec["id"])
    kw = dict(torch_dtype=torch.float16, device_map="cuda")
    if spec["quant4"]:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    model = AutoModelForCausalLM.from_pretrained(spec["id"], **kw).eval()
    return tok, model

def free(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()

@torch.no_grad()
def generate(tok, model, prompt, max_new):
    text = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    torch.cuda.synchronize()
    with PowerMeter() as pm:
        t0 = time.perf_counter()
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False)
        torch.cuda.synchronize()
        latency = time.perf_counter() - t0
    new_ids = out[0][inputs["input_ids"].shape[1]:]
    return {
        "response": tok.decode(new_ids, skip_special_tokens=True),
        "latency_s": latency,
        "energy_j": pm.energy_j,
        "mean_w": pm.mean_w,
        "in_tokens": int(inputs["input_ids"].shape[1]),
        "out_tokens": int(len(new_ids)),
    }

def parse_mc(resp):
    m = re.search(r"\b([ABCD])\b", resp)
    return m.group(1) if m else None

def parse_gsm(resp):
    m = re.search(r"Answer:\s*\$?\s*(-?[\d,]*\.?\d+)", resp)
    if not m:
        nums = re.findall(r"-?[\d,]*\.?\d+", resp)
        if not nums:
            return None
        s = nums[-1]
    else:
        s = m.group(1)
    return s.replace(",", "").rstrip(".")

def is_correct(q, resp):
    pred = parse_mc(resp) if q["type"] == "mc" else parse_gsm(resp)
    if pred is None:
        return pred, False
    if q["type"] == "mc":
        return pred, pred == q["gold"]
    try:
        return pred, abs(float(pred) - float(q["gold"])) < 1e-4
    except ValueError:
        return pred, False

## 4. Main run
Loads one tier at a time, runs every query that is not yet in `results.jsonl`, and appends each result immediately.
Batch size is 1 on purpose: it gives clean per-query latency and energy numbers.

In [ ]:
def load_done():
    done = set()
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH) as f:
            for line in f:
                r = json.loads(line)
                done.add((r["tier"], r["qid"]))
    return done

done = load_done()

for spec in TIERS:
    todo = [q for q in queries if (spec["name"], q["qid"]) not in done]
    if not todo:
        print(f"[{spec['name']}] already complete"); continue
    print(f"[{spec['name']}] loading {spec['id']} ... {len(todo)} queries to run")
    tok, model = load_model(spec)

    generate(tok, model, "Say hi.", 4)                 # warm-up (not logged)
    idle_w = measure_idle_watts(3.0)

    t_start = time.time()
    with open(RESULTS_PATH, "a") as f:
        for i, q in enumerate(todo):
            max_new = MAX_NEW_GSM if q["type"] == "math" else MAX_NEW_MC
            g = generate(tok, model, q["prompt"], max_new)
            pred, ok = is_correct(q, g["response"])
            row = {
                "tier": spec["name"], "model": spec["id"], "qid": q["qid"], "dataset": q["dataset"],
                "gold": q["gold"], "pred": pred, "correct": bool(ok),
                "latency_s": g["latency_s"], "energy_j": g["energy_j"],
                "energy_net_j": max(g["energy_j"] - idle_w * g["latency_s"], 0.0),
                "mean_w": g["mean_w"], "idle_w": idle_w,
                "in_tokens": g["in_tokens"], "out_tokens": g["out_tokens"],
                "response": g["response"][:600],
            }
            f.write(json.dumps(row) + "\n"); f.flush()
            if (i + 1) % 10 == 0:
                print(f"  {i+1}/{len(todo)}  ({(time.time()-t_start)/60:.1f} min elapsed)")
    free(model); del tok
    print(f"[{spec['name']}] done")

print("All tiers finished.")

## 5. Quick look at the results + oracle router upper bound

In [ ]:
df = pd.read_json(RESULTS_PATH, lines=True)
order = [t["name"] for t in TIERS]
df["tier"] = pd.Categorical(df["tier"], categories=order, ordered=True)

print("== Accuracy by tier and dataset ==")
print(df.pivot_table(index="tier", columns="dataset", values="correct", aggfunc="mean", observed=True).round(3))

print("\n== Mean cost per query by tier ==")
print(df.groupby("tier", observed=True)[["latency_s", "energy_j", "energy_net_j", "out_tokens"]].mean().round(3))

# Oracle router: for each query, pick the cheapest tier that answers correctly (else the largest tier).
wide_ok = df.pivot(index="qid", columns="tier", values="correct")
wide_e = df.pivot(index="qid", columns="tier", values="energy_net_j")
wide_ok, wide_e = wide_ok.dropna(), wide_e.loc[wide_ok.dropna().index]

def oracle_pick(row):
    for t in order:
        if row[t]:
            return t
    return order[-1]

pick = wide_ok.apply(oracle_pick, axis=1)
oracle_energy = sum(wide_e.loc[q, pick[q]] for q in wide_ok.index) / len(wide_ok)
oracle_acc = sum(wide_ok.loc[q, pick[q]] for q in wide_ok.index) / len(wide_ok)
large_energy, large_acc = wide_e[order[-1]].mean(), wide_ok[order[-1]].mean()

print("\n== Oracle routing (upper bound for your router) ==")
print(pick.value_counts().to_dict())
print(f"Always-large : acc={large_acc:.3f}  energy={large_energy:.2f} J/query")
print(f"Oracle router: acc={oracle_acc:.3f}  energy={oracle_energy:.2f} J/query  "
      f"({100*(1-oracle_energy/large_energy):.0f}% less energy)")

## Notes and caveats (write these in your report)
- **Short MC answers (about 3-8 tokens) finish in under a second**, so their energy readings are noisy. Compare *averages over many queries*, not single rows.
- NVML reports **GPU power only** (not CPU/RAM), which is fine as long as you say so.
- `energy_net_j` subtracts idle power. Report both raw and net.
- Greedy decoding is used, so each (query, tier) outcome is deterministic and reproducible.
- **Next (Week 2):** embed each query (sentence-transformers), train a classifier to predict the cheapest correct tier, and compare it with always-small, always-large, random, and length-based baselines.